## Project 2: Privacy-Preserving Health Diagnostics

### This project demonstrates how to train a PyTorch neural network under strict mathematical Differential Privacy guarantees using Meta's opacus engine.

In [1]:
import torch
import opacus
import sklearn
import pandas as pd

In [2]:
print(f"PyTorch Version: {torch.__version__}")
print(f"Opacus Version: {opacus.__version__}")
print("Environment is perfectly staged for Privacy Engineering!")

PyTorch Version: 2.2.2
Opacus Version: 1.4.0
Environment is perfectly staged for Privacy Engineering!


### We are going to use the classic Breast Cancer dataset from Scikit-Learn.

### There are two major rules when feeding data into PyTorch:

Scaling: Neural networks use Calculus (gradients) to learn. If one column has values in the millions (like Salary) and another column has values in the decimals (like BMI), the gradients will explode and the network will crash. We must force every column to have a mean of 0 and a standard deviation of 1. We use Scikit-Learn's StandardScaler for this.
Splitting: We must separate our data into X_train (the data the AI studies) and X_test (the data we use to grade the AI).

In [3]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
# Load the raw data
data = load_breast_cancer()


In [31]:
print(data.feature_names)

['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']


#### X (The Features): These are the inputs the AI studies. In this dataset, there are 30 different medical measurements (like tumor radius, texture, area, etc.). Because it is a grid of 30 columns and 569 rows, it is 2-Dimensional. That is why it must be stored as a pd.DataFrame (a 2D table).
#### y (The Target): This is the answer key. It is just a single column of 1s and 0s (1 = Cancer, 0 = No Cancer). Because it is only a single column, it is 1-Dimensional. That is why it must be stored as a pd.Series (a 1D list).

In [6]:
x = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

#### Split the data into Training (80%) and Testing (20%)

In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42)

#### Initialize the Scaler

In [8]:
scaler = StandardScaler()

### Scale the features to prevent exploding gradients

In [9]:
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_test)
print(f"Training data shape: {x_train_scaled.shape}")
print(f"Testing data shape: {x_test_scaled.shape}")

Training data shape: (455, 30)
Testing data shape: (114, 30)


#### PyTorch does not understand Pandas. It only understands its own proprietary object called a Tensor (which is highly optimized for GPU calculus). We need to explicitly translate our data so PyTorch can digest it.

#### Furthermore, feeding all 455 patients into the AI simultaneously is computationally heavy. We use a DataLoader to shuffle the patients and feed them into the network in small "batches" of 32 at a time.

In [10]:
from torch.utils.data import TensorDataset, DataLoader


#### Cast X arrays to PyTorch Tensors

In [11]:
x_train_tensor = torch.tensor(x_train_scaled, dtype=torch.float32)
x_test_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)

#### Cast y arrays to PyTorch Tensors, AND explicitly reshape them to (N, 1)

In [12]:
from numpy import float32
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

#### Glue them together so PyTorch knows which X belongs to which y

In [13]:
train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

#### Create the DataLoaders to feed the AI in chunks

In [14]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [15]:
print(f"Data successfully wrapped! Number of training batches: {len(train_loader)}")

Data successfully wrapped! Number of training batches: 15


## A Neural Network is just a sequence of layers doing matrix multiplications.

### Input Layer: Must have exactly the same number of input nodes as you have medical features (you printed out 30 features earlier).
### Hidden Layers: We use nn.ReLU() (Rectified Linear Unit) activation functions to allow the network to learn complex, non-linear patterns.
### Output Layer: Must output exactly 1 node (because we are doing Binary Classification: Cancer or No Cancer).
### Important Note: We do NOT put a Sigmoid() activation at the very end of the network. PyTorch's loss function (BCEWithLogitsLoss) calculates the calculus much faster and safer if we let it handle the Sigmoid internally.

In [16]:
import torch.nn as nn

#### Build the Neural Network Architecture

In [17]:
model = nn.Sequential(
    # Layer 1
    nn.Linear(in_features=30, out_features=16),
    nn.ReLU(),

    # Layer 2

    nn.Linear(in_features=16, out_features=8),
    nn.ReLU(),

    # Layer 3(Final Output)

    nn.Linear(in_features=8, out_features=1)
)
print("Neural Network Architecture Built Successfully!")
print(model)

Neural Network Architecture Built Successfully!
Sequential(
  (0): Linear(in_features=30, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=1, bias=True)
)


### The Concept
To train a network, you need three things:

The Model: You just built this!
The Loss Function (criterion): We use BCEWithLogitsLoss (Binary Cross Entropy). The "With Logits" part is exactly why we didn't use a Sigmoid activation in the previous step; PyTorch applies it automatically here because it is mathematically more stable for calculus calculation.
The Optimizer: We use Adam, which looks at the Loss and nudges the model's weights to be slightly better on the next try.

In [18]:
import torch.optim as optim

In [19]:
# Define the Loss Function and Optimizer

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [20]:
# Standard PyTorch training Loop
epochs = 20


Think of Step 4 as a student trying to pass a very difficult medical exam. Here is what every single line of that training loop is actually doing in plain English:

1. optimizer.zero_grad() (Erasing the Whiteboard)
PyTorch has a weird quirk: it automatically hoards all the calculus calculations from the previous batch. If we don't erase the whiteboard before the next batch, the math overlaps and the model gets incredibly confused. This line simply wipes the memory clean for a fresh start.

2. outputs = model(X_batch) (Taking the Test)
The model looks at 32 patients (X_batch) and blindly guesses whether they have cancer or not based on its current, random brain state.

3. loss = criterion(outputs, y_batch) (Grading the Test)
The teacher (our Loss Function) takes the model's blind guesses (outputs) and compares them to the true answers (y_batch). The output is a single number called the Loss—which represents exactly how terribly the model failed.

4. loss.backward() (Backpropagation)
This is the magic of Deep Learning. By running .backward(), PyTorch triggers a massive chain of Calculus derivatives. It traces backwards through every single neuron in the network and figures out exactly which neuron's fault it was that the guess was wrong.

5. optimizer.step() (Studying)
Now that we know who to blame, the Optimizer (Adam) steps in. It physically reaches into the neural network and nudges the weights of the bad neurons slightly down, and the good neurons slightly up. On the next batch, the model will be infinitesimally smarter!

The Final Exam (Testing Phase)
At the bottom of the code, you see model.eval() and torch.no_grad(). When we feed the hidden X_test data into the model, we want to see how smart it really is. We turn off the gradients (no_grad) so the model cannot use the optimizer to study and cheat on the final exam!

In [21]:
for epochs in range(epochs):
    model.train()

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()               # Clear old gradients
        outputs = model(x_batch)            # Forward pass (x_batch)
        loss = criterion(outputs, y_batch)  # Calculate how wrong it is
        loss.backward()                     # Backward pass (Calculus)
        optimizer.step()                    # Update the weights

# Grade the AI on the hidden test data
model.eval() # Tells PyTorch we are grading it now
with torch.no_grad():
    test_predrictions = model(x_test_tensor)
    # Convert raw logits to 0 or 1
    test_predrictions = torch.round(torch.sigmoid(test_predrictions))

    # Calculate Accuracy

    correct = (test_predrictions == y_test_tensor).sum().item()

    accuracy = correct / len(y_test_tensor)

print(f"Baseline (Non-Private) Test Accuracy: {accuracy * 100:.2f}%")


Baseline (Non-Private) Test Accuracy: 99.12%


## Step 5: DP-SGD Training.

#### Because our first model (model) already memorized the patients, it is permanently tainted. We must build a brand new brain, and immediately attach the Privacy Engine to it before it ever looks at the data.

In [22]:
from opacus import PrivacyEngine

Build a brand new brain! (The old one is permanently tainted)

In [23]:
private_model = nn.Sequential(
    nn.Linear(30, 16), nn.ReLU(),
    nn.Linear(16, 8), nn.ReLU(),
    nn.Linear(8, 1)
)

# Create a new optimizer for the private brain
private_optimizer = optim.Adam(private_model.parameters(), lr=0.001)

# Initiate Meta's Privacy Engine
privacy_engine = PrivacyEngine()

private_model, private_optimizer, private_train_loader = privacy_engine.make_private(
    module=private_model,
    optimizer=private_optimizer,
    data_loader=train_loader,
    noise_multiplier=1.0,
    max_grad_norm=1.0
)
print("Privacy Engine successfully attached! The model is now protected by DP-SGD")

Privacy Engine successfully attached! The model is now protected by DP-SGD


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/opacus/privacy_engine.py:142: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


The training loop for a Differential Privacy model is functionally identical to the standard loop. You still zero the gradients, calculate loss, and step the optimizer.

The only difference is that Opacus does all the heavy lifting in the background. Because it tracks all the noise it injects, we can ask the engine exactly what our Privacy Budget ($\epsilon$) is at the end of every epoch.

We must also define $\delta$ (Delta). Delta is the catastrophic failure rate—the percentage chance that the mathematical privacy completely fails and a patient is exposed. By industry standards, $\delta$ should be less than $1/N$ (where $N$ is the number of patients in the dataset). Since we have 455 training patients, we will set $\delta = 10^{-5}$ (which is written in Python as 1e-5).

In [24]:
import warnings
warnings.filterwarnings('ignore') # Opacus throws some harmless warnings we can ignore

epochs = 20

print("Starting Mathematically Private Training...")

for epoch in range(epochs):
    private_model.train()
    
    for x_batch, y_batch in train_loader:
        private_optimizer.zero_grad()
        
        # Forward Pass
        outputs = private_model(x_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward Pass (Opacus secretly clips and injects noise here!)
        loss.backward()
        private_optimizer.step()
        
    # Calculate the Privacy Budget Epsilon at the end of the epoch!
    epsilon = privacy_engine.get_epsilon(delta=1e-5)
    
    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f} | Epsilon: {epsilon:.2f}")

# Grade the Private AI on the hidden test data
private_model.eval()
with torch.no_grad():
    private_preds = private_model(x_test_tensor)
    private_preds = torch.round(torch.sigmoid(private_preds))
    correct = (private_preds == y_test_tensor).sum().item()
    private_accuracy = correct / len(y_test_tensor)

print(f"\nPrivate Model Test Accuracy: {private_accuracy * 100:.2f}%")
print(f"Final Privacy Guarantee: (ε = {epsilon:.2f}, δ = 1e-5)")


Starting Mathematically Private Training...
Epoch 0 | Loss: 0.7095 | Epsilon: 2.33
Epoch 5 | Loss: 0.5711 | Epsilon: 4.47
Epoch 10 | Loss: 0.4973 | Epsilon: 5.89
Epoch 15 | Loss: 0.3143 | Epsilon: 7.08
Epoch 19 | Loss: 0.1057 | Epsilon: 7.93

Private Model Test Accuracy: 96.49%
Final Privacy Guarantee: (ε = 7.93, δ = 1e-5)


### Step 7: Membership Inference Attack (MIA)

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

model.eval() # Ensure baseline model is in grading mode
with torch.no_grad():
    # 1. Hacker extracts Confidence Scores from the Training Data
    train_confidences = torch.sigmoid(model(x_train_tensor)).numpy()
    
    # 2. Hacker extracts Confidence Scores from the Hidden Test Data
    test_confidences = torch.sigmoid(model(x_test_tensor)).numpy()

# 3. Combine them into one giant dataset for the Hacker to study
X_attack = np.vstack((train_confidences, test_confidences))

# 4. Create the Answer Key (1 = Train Data, 0 = Test Data)
y_attack_train = np.ones(len(train_confidences))
y_attack_test = np.zeros(len(test_confidences))
y_attack = np.concatenate((y_attack_train, y_attack_test))

# 5. Initialize the Hacker's Attack Model (A simple Logistic Regression)
attack_model = LogisticRegression()

# 6. Train the Hacker to spot the difference between memorized vs hidden data!
attack_model.fit(X_attack, y_attack)

# 7. Grade the Hacker
hacker_predictions = attack_model.predict(X_attack)
hacker_accuracy = accuracy_score(y_attack, hacker_predictions)

print(f"Hacker MIA Accuracy on Baseline Model: {hacker_accuracy * 100:.2f}%")


Hacker MIA Accuracy on Baseline Model: 79.96%


Because the Hacker scored 79.96%, it means they are successfully looking at the PyTorch confidence scores and accurately reverse-engineering the identity of your patients 8 out of 10 times. In a healthcare or finance setting, your company would be facing a multi-million dollar HIPAA fine.

This proves exactly why the Baseline model's 99.12% accuracy was a trap. It didn't learn general medicine; it memorized patients.

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

# 1. Randomly shuffle and slice the train confidences so there are only 114 of them!
np.random.shuffle(train_confidences)
balanced_train_confidences = train_confidences[:len(test_confidences)]

# 2. Combine the perfectly balanced 50/50 dataset
X_attack_balanced = np.vstack((balanced_train_confidences, test_confidences))

# 3. Create the 50/50 Answer Key
y_attack_balanced = np.concatenate((
    np.ones(len(balanced_train_confidences)), 
    np.zeros(len(test_confidences))
))

# 4. Train the Hacker again!
attack_model.fit(X_attack_balanced, y_attack_balanced)
hacker_accuracy = accuracy_score(y_attack_balanced, attack_model.predict(X_attack_balanced))

print(f"True Hacker MIA Accuracy: {hacker_accuracy * 100:.2f}%")



True Hacker MIA Accuracy: 51.75%
